In [ ]:
import requests
import time
from pyspark.sql import SparkSession

# --- 3. ADIMDA ALDIĞINIZ DEĞERLERİ YAZIN ---
client_id = "polaris_root"
client_secret = "polaris_secret123"

base_url = "http://polaris-catalog-service.bigdata.svc.cluster.local:8181"
localstack_url = "http://localstack-service.bigdata.svc.cluster.local:4566"

# 1. Token Al
token_resp = requests.post(
    f"{base_url}/api/catalog/v1/oauth/tokens",
    data={"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret, "scope": "PRINCIPAL_ROLE:ALL"},
    headers={"Polaris-Realm": "POLARIS"}
)
token = token_resp.json().get("access_token")
auth_headers = {"Authorization": f"Bearer {token}", "Polaris-Realm": "POLARIS"}

# 2. Katalog Yapılandırması (storage-check bypass özellikleri ile)
catalog_payload = {
    "catalog": {
        "name": "polaris",
        "type": "INTERNAL",
        "properties": {
            "default-base-location": "s3://warehouse",
            "s3.endpoint": localstack_url,
            "s3.path-style-access": "true",
            "client.region": "us-east-1"
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": ["s3://warehouse", "s3://warehouse/*"],
            "endpoint": localstack_url,
            "pathStyleAccess": True,
            "stsUnavailable": True,
            "region": "us-east-1"
        }
    }
}

get_resp = requests.get(f"{base_url}/api/management/v1/catalogs/polaris", headers=auth_headers)
if get_resp.status_code == 200:
    catalog_payload["currentEntityVersion"] = get_resp.json().get("entityVersion", 1)
    requests.put(f"{base_url}/api/management/v1/catalogs/polaris", json=catalog_payload, headers=auth_headers)
else:
    requests.post(f"{base_url}/api/management/v1/catalogs", json=catalog_payload, headers=auth_headers)

# Yetkileri ver
for priv in ["CATALOG_MANAGE_CONTENT", "CATALOG_MANAGE_ACCESS", "TABLE_WRITE_DATA", "TABLE_READ_DATA"]:
    requests.put(
        f"{base_url}/api/management/v1/catalogs/polaris/catalog-roles/catalog_admin/grants",
        json={"grant": {"type": "catalog", "privilege": priv}},
        headers=auth_headers
    )
print("✅ Polaris kataloğu hazırlandı.")


In [ ]:
import requests
import time
from pyspark.sql import SparkSession

# --- ŞİFRELERİNİZ ---
client_id = "polaris_root"
client_secret = "polaris_secret123"

base_url = "http://polaris-catalog-service.bigdata.svc.cluster.local:8181"
localstack_url = "http://localstack-service.bigdata.svc.cluster.local:4566"

# 1. Spark Oturumu (s3:// ve s3a:// protokollerinin her ikisi de tanımlı)
print("Spark oturumu başlatılıyor...")
spark = SparkSession.builder \
    .appName("Iceberg-Polaris-Final") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.5.0"
    ) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.polaris.type", "rest") \
    .config("spark.sql.catalog.polaris.uri", f"{base_url}/api/catalog") \
    .config("spark.sql.catalog.polaris.credential", f"{client_id}:{client_secret}") \
    .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL") \
    .config("spark.sql.catalog.polaris.warehouse", "polaris") \
    .config("spark.sql.catalog.polaris.header.X-Iceberg-Access-Delegation", "false") \
    .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.polaris.s3.endpoint", localstack_url) \
    .config("spark.sql.catalog.polaris.s3.path-style-access", "true") \
    .config("spark.sql.catalog.polaris.s3.access-key-id", "test") \
    .config("spark.sql.catalog.polaris.s3.secret-access-key", "test") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.endpoint", localstack_url) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.defaultCatalog", "polaris") \
    .getOrCreate()


print("✅ Spark oturumu açıldı!\n")

# 4. Namespace ve Tablo Oluşturma
print("4. Namespace ve Tablo oluşturuluyor...")
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.ts_db")

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.ts_db.users (
        id BIGINT,
        isim STRING,
        meslek STRING
    ) USING iceberg
""")
print("✅ Tablo 'polaris.ts_db.users' başarıyla oluşturuldu!\n")

# 5. Veri Yazma (INSERT)
print("5. Veriler yazılıyor...")
spark.sql("""
    INSERT INTO polaris.ts_db.users VALUES 
    (1, 'Aleyna', 'Data Engineer'),
    (2, 'Ahmet', 'Backend Developer')
""")
print("✅ Veriler S3'e başarıyla yazıldı!\n")

# 6. Veri Okuma (SELECT)
print("--- Polaris Iceberg Tablosu: polaris.ts_db.users ---")
spark.sql("SELECT * FROM polaris.ts_db.users").show()



# Create New Polaris User

In [ ]:
import requests

# 1. Root ile Admin Token'ı Al
root_id = "bef09777670113ab"
root_secret = "a3bb4ae71ce4eba0ab69e17eb775e362"
base_url = "http://polaris-catalog-service.bigdata.svc.cluster.local:8181"

token_resp = requests.post(
    f"{base_url}/api/catalog/v1/oauth/tokens",
    data={
        "grant_type": "client_credentials",
        "client_id": root_id,
        "client_secret": root_secret,
        "scope": "PRINCIPAL_ROLE:ALL"
    },
    headers={"Polaris-Realm": "POLARIS"}
)
admin_token = token_resp.json().get("access_token")
headers = {"Authorization": f"Bearer {admin_token}", "Polaris-Realm": "POLARIS"}

# 2. Yeni Principal (Kullanıcı) Oluştur: spark_user
new_user_payload = {
    "principal": {
        "name": "spark_user"
    }
}
user_resp = requests.post(f"{base_url}/api/management/v1/principals", json=new_user_payload, headers=headers)

if user_resp.status_code == 201:
    user_data = user_resp.json()
    spark_client_id = user_data["principal"]["clientId"]
    spark_client_secret = user_data["principal"]["credential"]
    print(f"✅ Yeni Kullanıcı Oluşturuldu:")
    print(f"   Client ID: {spark_client_id}")
    print(f"   Client Secret: {spark_client_secret}")
else:
    print(f"Kullanıcı oluşturma yanıtı: {user_resp.text}")

# 3. Yeni Principal Role Oluştur: spark_principal_role
role_payload = {
    "principalRole": {
        "name": "spark_principal_role"
    }
}
requests.post(f"{base_url}/api/management/v1/principal-roles", json=role_payload, headers=headers)

# 4. Rolü Kullanıcıya Bağla (Principal -> Principal Role)
requests.put(
    f"{base_url}/api/management/v1/principals/spark_user/principal-roles",
    json={"principalRole": {"name": "spark_principal_role"}},
    headers=headers
)

# 5. Katalog Yetkisini Bu Role Bağla (Principal Role -> Catalog Role)
requests.put(
    f"{base_url}/api/management/v1/principal-roles/spark_principal_role/catalog-roles/polaris",
    json={"catalogRole": {"name": "catalog_admin"}},
    headers=headers
)

print("✅ 'spark_user' kullanıcısına 'polaris' kataloğu için tam yetki tanımlandı!")